In [ ]:
# %pip install catboost optuna

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import KFold, StratifiedKFold, train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer

import xgboost as xgb
import lightgbm as lgb
import catboost as cb

import optuna
from optuna.samplers import TPESampler

import os
# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

train.head()

,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,X11,Y
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,FDX07,19.20,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,NaN,Tier 3,Grocery Store,732.3800
4,NCD19,8.93,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052


In [ ]:
# Check missing values
train.isnull().sum()
test.isnull().sum()

,0
X1,0
X2,457
X3,0
X4,0
X5,0
X6,0
X7,0
X8,0
X9,699
X10,0


In [ ]:
column_mapping = {
    'X1': 'Item_Identifier',
    'X2': 'Item_Weight',
    'X3': 'Item_Fat_Content',
    'X4': 'Item_Visibility',
    'X5': 'Item_Type',
    'X6': 'Item_MRP',
    'X7': 'Outlet_Identifier',
    'X8': 'Outlet_Establishment_Year',
    'X9': 'Outlet_Size',
    'X10': 'Outlet_Location_Type',
    'X11': 'Outlet_Type',
    'Y': 'Item_Outlet_Sales'
}

train.rename(columns=column_mapping, inplace=True)
test.rename(columns=column_mapping, inplace=True)

print("Train columns:", train.columns.tolist())
print("Test columns:", test.columns.tolist())
train.head()

Train columns: ['Item_Identifier', 'Item_Weight', 'Item_Fat_Content', 'Item_Visibility', 'Item_Type', 'Item_MRP', 'Outlet_Identifier', 'Outlet_Establishment_Year', 'Outlet_Size', 'Outlet_Location_Type', 'Outlet_Type', 'Item_Outlet_Sales']
Test columns: ['Item_Identifier', 'Item_Weight', 'Item_Fat_Content', 'Item_Visibility', 'Item_Type', 'Item_MRP', 'Outlet_Identifier', 'Outlet_Establishment_Year', 'Outlet_Size', 'Outlet_Location_Type', 'Outlet_Type']


,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,FDX07,19.20,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,NaN,Tier 3,Grocery Store,732.3800
4,NCD19,8.93,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052


In [ ]:
train['Item_Fat_Content'] = train['Item_Fat_Content'].replace({
    'LF': 'Low Fat',
    'low fat': 'Low Fat',
    'reg': 'Regular'
})
test['Item_Fat_Content'] = test['Item_Fat_Content'].replace({
    'LF': 'Low Fat',
    'low fat': 'Low Fat',
    'reg': 'Regular'
})

print(train['Item_Fat_Content'].value_counts())

Item_Fat_Content
Low Fat    3889
Regular    2111
Name: count, dtype: int64


In [ ]:
outlet_size_mode = train.groupby('Outlet_Type')['Outlet_Size'].agg(lambda x: x.mode()[0] if not x.mode().empty else 'Medium')
train['Outlet_Size'] = train.apply(
    lambda row: outlet_size_mode[row['Outlet_Type']] if pd.isnull(row['Outlet_Size']) else row['Outlet_Size'],
    axis=1
)
test['Outlet_Size'] = test.apply(
    lambda row: outlet_size_mode.get(row['Outlet_Type'], 'Medium') if pd.isnull(row['Outlet_Size']) else row['Outlet_Size'],
    axis=1
)

print(train.isnull().sum())
print(test.isnull().sum())

Item_Identifier                 0
Item_Weight                  1006
Item_Fat_Content                0
Item_Visibility                 0
Item_Type                       0
Item_MRP                        0
Outlet_Identifier               0
Outlet_Establishment_Year       0
Outlet_Size                     0
Outlet_Location_Type            0
Outlet_Type                     0
Item_Outlet_Sales               0
dtype: int64
Item_Identifier                0
Item_Weight                  457
Item_Fat_Content               0
Item_Visibility                0
Item_Type                      0
Item_MRP                       0
Outlet_Identifier              0
Outlet_Establishment_Year      0
Outlet_Size                    0
Outlet_Location_Type           0
Outlet_Type                    0
dtype: int64


In [ ]:
current_year = 2025  # You can also use max year in data (train['Outlet_Establishment_Year'].max())
train['Outlet_Age'] = current_year - train['Outlet_Establishment_Year']
test['Outlet_Age'] = current_year - test['Outlet_Establishment_Year']

In [ ]:
def target_encode(train, test, col, target, folds=5):
    """Apply target encoding with K-fold on train, and map to test using global mean."""
    train_encoded = np.zeros(train.shape[0])
    kf = KFold(n_splits=folds, shuffle=True, random_state=RANDOM_STATE)
    for tr_idx, val_idx in kf.split(train):
        tr, val = train.iloc[tr_idx], train.iloc[val_idx]
        means = tr.groupby(col)[target].mean()
        train_encoded[val_idx] = val[col].map(means).fillna(tr[target].mean())
    # Global mean for test
    global_mean = train.groupby(col)[target].mean()
    test_encoded = test[col].map(global_mean).fillna(train[target].mean())
    return train_encoded, test_encoded

In [ ]:
# 'Item_Identifier' – very high cardinality, but we can encode
train['Item_Id_Enc'], test['Item_Id_Enc'] = target_encode(train, test, 'Item_Identifier', 'Item_Outlet_Sales')

# 'Outlet_Identifier' – also high
train['Outlet_Id_Enc'], test['Outlet_Id_Enc'] = target_encode(train, test, 'Outlet_Identifier', 'Item_Outlet_Sales')

In [ ]:
cat_cols = ['Item_Fat_Content', 'Item_Type', 'Outlet_Size', 'Outlet_Location_Type', 'Outlet_Type']
train = pd.get_dummies(train, columns=cat_cols, drop_first=True)
test = pd.get_dummies(test, columns=cat_cols, drop_first=True)

# Ensure test has same columns as train (align)
train_cols = train.columns
test = test.reindex(columns=train_cols.drop('Item_Outlet_Sales'), fill_value=0)

In [ ]:
train['Item_MRP_Outlet_Age'] = train['Item_MRP'] * train['Outlet_Age']
test['Item_MRP_Outlet_Age'] = test['Item_MRP'] * test['Outlet_Age']

train['Item_Visibility_Weight'] = train['Item_Visibility'] * train['Item_Weight']
test['Item_Visibility_Weight'] = test['Item_Visibility'] * test['Item_Weight']

In [ ]:
train['Visibility_Zero'] = (train['Item_Visibility'] == 0).astype(int)
test['Visibility_Zero'] = (test['Item_Visibility'] == 0).astype(int)
y = train['Item_Outlet_Sales']
y_log = np.log1p(y)

In [ ]:
X = train.drop(['Item_Outlet_Sales', 'Item_Identifier', 'Outlet_Identifier', 'Outlet_Establishment_Year'], axis=1)
X_test = test.drop(['Item_Identifier', 'Outlet_Identifier', 'Outlet_Establishment_Year'], axis=1, errors='ignore')

In [ ]:
def objective_xgb(trial, X, y, cv=5):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 2000, step=100),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'random_state': RANDOM_STATE,
        'tree_method': 'hist',
        'early_stopping_rounds': 50  # Moved here from fit()
    }

    kf = KFold(n_splits=cv, shuffle=True, random_state=RANDOM_STATE)
    scores = []
    for train_idx, val_idx in kf.split(X):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = xgb.XGBRegressor(**params)
        # Removed early_stopping_rounds from fit()
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        pred = model.predict(X_val)
        score = np.sqrt(mean_squared_error(y_val, pred))
        scores.append(score)

    return np.mean(scores)

study_xgb = optuna.create_study(direction='minimize', sampler=TPESampler(seed=RANDOM_STATE))
study_xgb.optimize(lambda trial: objective_xgb(trial, X, y), n_trials=30, show_progress_bar=True)

print('Best XGBoost trial:', study_xgb.best_trial.params)
print('Best XGBoost CV RMSE:', study_xgb.best_value)

[I 2026-02-20 23:36:19,558] A new study created in memory with name: no-name-859f1a9c-7a33-4a28-a85a-1b083ca8b09f


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-02-20 23:36:24,167] Trial 0 finished with value: 1139.9781908711132 and parameters: {'n_estimators': 1000, 'max_depth': 12, 'learning_rate': 0.1205712628744377, 'subsample': 0.7993292420985183, 'colsample_bytree': 0.5780093202212182, 'gamma': 0.7799726016810132, 'reg_alpha': 3.3323645788192616e-08, 'reg_lambda': 0.6245760287469893, 'min_child_weight': 7}. Best is trial 0 with value: 1139.9781908711132.
[I 2026-02-20 23:36:25,095] Trial 1 finished with value: 1115.4368337256776 and parameters: {'n_estimators': 1600, 'max_depth': 3, 'learning_rate': 0.2708160864249968, 'subsample': 0.9162213204002109, 'colsample_bytree': 0.6061695553391381, 'gamma': 0.9091248360355031, 'reg_alpha': 4.4734294104626844e-07, 'reg_lambda': 5.472429642032198e-06, 'min_child_weight': 6}. Best is trial 1 with value: 1115.4368337256776.
[I 2026-02-20 23:36:26,565] Trial 2 finished with value: 1105.8052801816334 and parameters: {'n_estimators': 1100, 'max_depth': 5, 'learning_rate': 0.08012737503998542, '

In [ ]:
def objective_lgb(trial, X, y, cv=5):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 2000, step=100),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('feature_fraction', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'random_state': RANDOM_STATE,
        'verbosity': -1,
    }

    kf = KFold(n_splits=cv, shuffle=True, random_state=RANDOM_STATE)
    scores = []
    for train_idx, val_idx in kf.split(X):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = lgb.LGBMRegressor(**params)
        # Removed verbose=False which caused the TypeError
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(50)])
        pred = model.predict(X_val)
        score = np.sqrt(mean_squared_error(y_val, pred))
        scores.append(score)

    return np.mean(scores)

study_lgb = optuna.create_study(direction='minimize', sampler=TPESampler(seed=RANDOM_STATE))
study_lgb.optimize(lambda trial: objective_lgb(trial, X, y), n_trials=30, show_progress_bar=True)

print('Best LightGBM trial:', study_lgb.best_trial.params)
print('Best LightGBM CV RMSE:', study_lgb.best_value)

[I 2026-02-20 23:38:47,918] A new study created in memory with name: no-name-eb4e27f0-37fe-4507-9eea-bca72c9f03a4


  0%|          | 0/30 [00:00<?, ?it/s]

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[25]	valid_0's l2: 1.22142e+06
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[23]	valid_0's l2: 1.13362e+06
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[24]	valid_0's l2: 1.1156e+06
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[46]	valid_0's l2: 1.20954e+06
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[25]	valid_0's l2: 1.26278e+06
[I 2026-02-20 23:38:48,642] Trial 0 finished with value: 1089.929113036204 and parameters: {'n_estimators': 1000, 'max_depth': 15, 'learning_rate': 0.1205712628744377, 'subsample': 0.7993292420985183, 'feature_fraction': 0.5780093202212182, 'reg_alpha': 2.5348407664333426e-07, 'reg_lambda': 3.3323645788192616e-08, 'min_child_samples': 44}. Best is trial 0 with val

In [ ]:
def objective_cat(trial, X, y, cv=5):
    params = {
        'iterations': trial.suggest_int('iterations', 500, 2000, step=100),
        'depth': trial.suggest_int('depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-8, 10.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 1),
        'random_seed': RANDOM_STATE,
        'verbose': False,
    }

    kf = KFold(n_splits=cv, shuffle=True, random_state=RANDOM_STATE)
    scores = []
    for train_idx, val_idx in kf.split(X):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = cb.CatBoostRegressor(**params)
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val), early_stopping_rounds=50, verbose=False)
        pred = model.predict(X_val)
        score = np.sqrt(mean_squared_error(y_val, pred))
        scores.append(score)

    return np.mean(scores)

study_cat = optuna.create_study(direction='minimize', sampler=TPESampler(seed=RANDOM_STATE))
study_cat.optimize(lambda trial: objective_cat(trial, X, y), n_trials=30, show_progress_bar=True)

print('Best CatBoost trial:', study_cat.best_trial.params)
print('Best CatBoost CV RMSE:', study_cat.best_value)

[I 2026-02-20 23:39:20,255] A new study created in memory with name: no-name-f989c995-8045-498e-9162-eaf20595f75e


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-02-20 23:39:32,764] Trial 0 finished with value: 1087.500367799596 and parameters: {'iterations': 1000, 'depth': 10, 'learning_rate': 0.1205712628744377, 'l2_leaf_reg': 0.0024430162614261413, 'bagging_temperature': 0.15601864044243652}. Best is trial 0 with value: 1087.500367799596.
[I 2026-02-20 23:39:33,549] Trial 1 finished with value: 1081.9265861879778 and parameters: {'iterations': 700, 'depth': 3, 'learning_rate': 0.19030368381735815, 'l2_leaf_reg': 0.002570603566117598, 'bagging_temperature': 0.7080725777960455}. Best is trial 1 with value: 1081.9265861879778.
[I 2026-02-20 23:39:43,725] Trial 2 finished with value: 1090.0133526334826 and parameters: {'iterations': 500, 'depth': 10, 'learning_rate': 0.16967533607196555, 'l2_leaf_reg': 8.148018307012941e-07, 'bagging_temperature': 0.18182496720710062}. Best is trial 1 with value: 1081.9265861879778.
[I 2026-02-20 23:39:45,261] Trial 3 finished with value: 1080.6916202243235 and parameters: {'iterations': 700, 'depth': 5,

In [ ]:
def get_oof_predictions(model_class, params, X, y, n_folds=5):
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
    oof_preds = np.zeros(len(y))
    models = []
    for train_idx, val_idx in kf.split(X):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        if model_class == xgb.XGBRegressor:
            # Move early_stopping_rounds to the constructor
            model = xgb.XGBRegressor(**params, tree_method='hist', early_stopping_rounds=50)
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        elif model_class == lgb.LGBMRegressor:
            # Removed verbose=False which caused the TypeError
            model = lgb.LGBMRegressor(**params)
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(50)])
        elif model_class == cb.CatBoostRegressor:
            model = cb.CatBoostRegressor(**params)
            model.fit(X_tr, y_tr, eval_set=(X_val, y_val), early_stopping_rounds=50, verbose=False)

        oof_preds[val_idx] = model.predict(X_val)
        models.append(model)
    return oof_preds, models

In [ ]:
xgb_oof, xgb_models = get_oof_predictions(xgb.XGBRegressor, study_xgb.best_params, X, y)
lgb_oof, lgb_models = get_oof_predictions(lgb.LGBMRegressor, study_lgb.best_params, X, y)
cat_oof, cat_models = get_oof_predictions(cb.CatBoostRegressor, study_cat.best_params, X, y)

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[15]	valid_0's l2: 1.19831e+06
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[11]	valid_0's l2: 1.13012e+06
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[18]	valid_0's l2: 1.06704e+06
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[21]	valid_0's l2: 1.18391e+06
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[15]	valid_0's l2: 1.24508e+06


In [ ]:
print('XGB OOF RMSE:', np.sqrt(mean_squared_error(y, xgb_oof)))
print('LGB OOF RMSE:', np.sqrt(mean_squared_error(y, lgb_oof)))
print('Cat OOF RMSE:', np.sqrt(mean_squared_error(y, cat_oof)))

XGB OOF RMSE: 1101.9561853845444
LGB OOF RMSE: 1079.3012432662583
Cat OOF RMSE: 1078.5789472051554


In [ ]:
blend_oof = (xgb_oof + lgb_oof + cat_oof) / 3
print('Simple blend OOF RMSE:', np.sqrt(mean_squared_error(y, blend_oof)))

Simple blend OOF RMSE: 1081.609026322199


In [ ]:
from sklearn.linear_model import LinearRegression
stack_X = np.column_stack([xgb_oof, lgb_oof, cat_oof])
stack_model = LinearRegression()
stack_model.fit(stack_X, y)
weights = stack_model.coef_
print('Optimal weights:', weights)

weighted_oof = stack_model.predict(stack_X)
print('Weighted blend OOF RMSE:', np.sqrt(mean_squared_error(y, weighted_oof)))

Optimal weights: [-0.26591413  0.54269291  0.74621637]
Weighted blend OOF RMSE: 1074.9501806832156


In [ ]:
# XGBoost
xgb_final = xgb.XGBRegressor(**study_xgb.best_params, tree_method='hist', random_state=RANDOM_STATE)
xgb_final.fit(X, y)

# LightGBM
lgb_final = lgb.LGBMRegressor(**study_lgb.best_params, random_state=RANDOM_STATE, verbosity=-1)
lgb_final.fit(X, y)

# CatBoost
cat_final = cb.CatBoostRegressor(**study_cat.best_params, random_state=RANDOM_STATE, verbose=False)
cat_final.fit(X, y)

# Predict on test
xgb_test_pred = xgb_final.predict(X_test)
lgb_test_pred = lgb_final.predict(X_test)
cat_test_pred = cat_final.predict(X_test)

# Blend with optimal weights
stack_test = np.column_stack([xgb_test_pred, lgb_test_pred, cat_test_pred])
final_pred = stack_model.predict(stack_test)

In [ ]:
submission = pd.DataFrame({
    'Item_Identifier': test['Item_Identifier'],
    'Outlet_Identifier': test['Outlet_Identifier'],
    'Item_Outlet_Sales': final_pred
})
submission.to_csv('BYE.csv', index=False)
print('Submission saved.')

Submission saved.


# Task
Refactor the machine learning pipeline for the Big Mart Sales dataset in "train.csv" and "test.csv" by renaming generic columns (X1-X11, Y) to descriptive names: `Item_Identifier`, `Item_Weight`, `Item_Fat_Content`, `Item_Visibility`, `Item_Type`, `Item_MRP`, `Outlet_Identifier`, `Outlet_Establishment_Year`, `Outlet_Size`, `Outlet_Location_Type`, `Outlet_Type`, and `Item_Outlet_Sales`.

The task includes updating all subsequent steps: standardizing fat content labels, imputing missing `Item_Weight` and `Outlet_Size`, engineering features like `Outlet_Age` and target-encoded identifiers, and fine-tuning XGBoost, LightGBM, and CatBoost models using Optuna. Finally, generate a weighted ensemble prediction and save the results to "BYE.csv".

## Identify and Rename Columns

### Subtask:
Rename the generic column names X1-X11 and Y to descriptive names in both train and test DataFrames.


**Reasoning**:
I will rename the generic columns in the train and test DataFrames to descriptive names using the provided mapping to improve clarity and fix previous errors caused by incorrect column names.



In [ ]:
column_mapping = {
    'X1': 'Item_Identifier',
    'X2': 'Item_Weight',
    'X3': 'Item_Fat_Content',
    'X4': 'Item_Visibility',
    'X5': 'Item_Type',
    'X6': 'Item_MRP',
    'X7': 'Outlet_Identifier',
    'X8': 'Outlet_Establishment_Year',
    'X9': 'Outlet_Size',
    'X10': 'Outlet_Location_Type',
    'X11': 'Outlet_Type',
    'Y': 'Item_Outlet_Sales'
}

train.rename(columns=column_mapping, inplace=True)
test.rename(columns=column_mapping, inplace=True)

print("Train columns:", train.columns.tolist())
print("Test columns:", test.columns.tolist())
train.head()